In [8]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:

import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import (
    ExtraTreesClassifier,
    RandomForestClassifier
)
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score

df = pd.DataFrame({
    'Age': [26, 47, 27, 24, 26, 49, 42, 46, 47, 44, 43, 47, 21, 21, 25, 41, 20, 45, 21, 49, 43, 29, 26, 20, 27, 24, 42, 27, 47, 44, 42, 47, 45, 21, 24, 45, 21, 49, 23, 40, 25, 43, 23, 21, 23, 44, 43, 22, 49, 48, 26, 48, 47, 20, 48, 23, 45, 21, 42, 22, 28, 43, 47, 20, 20, 24, 21, 25, 23, 40, 42, 44, 23, 23, 47, 29, 42, 41, 41, 44, 23, 26, 29, 43, 40, 28, 22, 23, 25, 40, 28, 20, 41, 44, 23, 22, 46, 44, 40, 47, 26, 41, 27, 28, 24, 46, 41, 47, 24, 43, 21, 20, 49, 48, 21, 40, 45, 21, 48, 23, 22, 44, 42, 23, 24, 41, 25, 23, 43, 45, 43, 42, 21, 21, 45, 42, 20, 47, 29, 48, 45, 20, 47, 46, 21, 40, 42, 49, 46, 22, 25, 29, 46, 46, 20, 43, 42, 25, 44, 23, 43, 48, 24, 25, 44, 29, 44, 46, 25, 44, 43, 41, 28, 49, 40, 43, 45, 41, 43, 42, 41, 24, 24, 25, 48, 26, 45, 24, 42, 46, 42, 40, 48, 23, 46, 44, 27, 43, 23, 42, 25, 41, 28, 20, 24, 49, 22, 25, 21, 46, 24, 28, 21, 29, 48, 21, 48, 46, 48, 20, 47, 20, 41, 43, 46, 43, 43, 45, 49, 45, 21, 28, 48, 49, 41, 26, 46, 20, 24, 24, 25, 22, 43, 47, 46, 44, 46, 29, 41, 45, 20, 42, 28, 44, 22, 48, 42, 20, 27, 21, 27, 26, 29, 29, 21, 49, 47, 22, 44, 43, 43, 47, 28, 40, 26, 48, 43, 49, 22, 41, 25, 41, 25, 43, 21, 21, 49, 21, 42, 49, 45, 42, 41, 47, 44, 40, 42, 49, 47, 29, 27, 49, 41, 46, 44, 44, 45, 48, 45, 24, 46, 22, 43, 41, 26, 45, 27, 48, 48, 23, 42, 28, 44, 22, 21, 47, 43, 22, 49, 42, 45, 29, 25, 24, 25, 20, 24, 28, 29, 21, 45, 29, 48, 44, 44, 28, 25, 27, 20, 47, 23, 20, 27, 20, 22, 44, 27, 49, 29, 46, 27, 21, 44, 44, 44, 26, 40, 26, 21, 43, 44, 22, 22, 41, 48, 41, 29, 26, 28, 20, 26, 25, 43, 28, 20, 23, 49, 23, 29, 41, 28, 21, 23, 25, 21, 44, 45, 41, 22, 29, 28, 49, 25, 23, 41, 21, 49, 47, 47, 28, 20, 24, 25, 24, 25, 25, 44, 23, 48, 26, 28, 26, 45, 48, 40, 48, 23, 48, 25, 21, 43, 41, 45, 43, 45, 42, 45, 41, 28, 48, 46, 29, 40, 24, 25, 46, 25, 23, 22, 23, 46, 23, 42, 20, 29, 25, 24, 42, 45, 20, 25, 47, 27, 29, 24, 42, 44, 49, 27, 21, 45, 44, 49, 48, 20, 41, 27, 43, 26, 45, 41, 44, 46, 24, 44, 28, 21, 22, 22, 44, 27, 25, 27, 43, 44, 43, 42, 24, 24, 43, 29, 23, 28, 43, 49, 21, 46, 49, 46, 46, 47, 25, 49, 26, 20, 43, 20, 20, 26, 46, 45, 27, 47, 49, 46, 45, 25, 25, 45, 48, 22, 27, 49, 23, 49, 48, 26, 23, 47, 40, 20, 44, 26, 41, 23, 25, 21, 46, 49, 29, 41, 25, 25, 49, 40, 23, 41, 44, 24, 23, 41, 22, 22, 44, 23, 29, 44, 42, 29, 27, 48, 48, 48, 41, 29, 46, 44, 21, 42, 20, 45, 46, 48, 24, 27, 47, 27, 29, 20, 29, 28, 25, 43, 47, 28, 40, 43, 42, 48, 24, 40, 45, 25, 26, 43, 40, 41, 44, 44, 47, 24, 47, 41, 28, 29, 23, 28, 45, 45, 41, 46, 41, 45, 29, 42, 21, 26, 43, 21, 45, 22, 43, 27, 48, 46, 49, 48, 46, 27, 20, 20, 42, 29, 42, 47, 20, 26, 43, 23, 47, 40, 48, 44, 43, 27, 20, 41, 47, 26, 22, 44, 22, 43, 27, 43, 21, 46, 22, 21, 48, 25, 41, 21, 48, 22, 42, 20, 46, 26, 43, 41, 46, 29, 42, 21, 45, 26, 43, 21, 26, 49, 20, 28, 42, 22, 27, 26, 42, 25, 43, 24, 22, 22, 41, 27, 21, 41, 27, 41, 49, 23, 41, 25, 45, 40, 21, 40, 45, 43, 42, 42, 24, 45, 22, 44, 21, 21, 44, 41, 43, 29, 43, 22, 26, 26, 41, 46, 24, 25, 44, 21, 41, 22, 48, 29, 41, 27, 48, 46, 27, 26, 42, 21, 48, 20, 46, 40, 43, 43, 46, 47, 25, 25, 23, 43, 43, 43, 45, 24, 25, 44, 24, 29, 46, 28, 46, 21, 45, 44, 46, 41, 29, 45, 45, 44, 25, 46, 28, 47, 27, 21, 40, 24, 49, 41, 44, 29, 24, 44, 43, 25, 42, 40, 44, 23, 22, 29, 49, 43, 41, 20, 41, 47, 42, 48, 21, 26, 49, 25, 25, 40, 48, 20, 47, 23, 48, 22, 41, 23, 20, 21, 49, 29, 21, 46, 41, 28, 27, 47, 40, 40, 46, 47, 28, 44, 22, 22, 21, 47, 22, 25, 46, 22, 24, 47, 20, 29, 28, 44, 43, 44, 22, 27, 43, 20, 26, 46, 42, 25, 27, 28, 44, 45, 47, 49, 29, 40, 47, 47, 48, 48, 24, 25, 42, 26, 43, 43, 28, 22, 48, 49, 46, 21, 49, 44, 43, 21, 41, 45, 20, 41, 49, 21, 47, 24, 27, 48, 48, 47, 49, 27, 41, 23, 43, 43, 44, 20, 40, 47, 44, 42, 29, 46, 26, 21, 22, 26, 23, 20, 47, 28, 27, 28, 45, 45, 26, 45, 24, 20, 44, 22, 41, 42, 23, 21, 45, 28, 20, 24, 42, 45, 28, 46, 46, 48, 44, 47, 25, 26, 27, 40, 44, 28, 40, 23, 23, 27, 49, 41, 24, 22, 29, 21, 41, 48, 28, 20, 43, 45, 27, 25, 24, 45, 40, 48, 20, 28, 48, 43, 29, 46, 45, 22, 42, 21, 20, 27, 22, 22, 25, 27, 45, 28, 23, 44, 45, 44, 27, 47, 28, 22, 28, 21, 43, 48, 27, 28, 24, 46, 27, 25, 47, 44, 29, 46, 43, 24, 47, 20, 41, 27, 22, 29, 40, 27, 45, 21, 27, 45, 25, 27, 42, 27, 20, 28, 28, 40, 41, 23, 40, 26, 45, 29, 28, 22, 26, 29, 29, 46, 26, 49, 23, 48, 43, 49, 20, 28, 40, 27, 45, 27, 42, 44, 22, 49, 29, 49, 42, 24, 25, 23, 40, 23, 49, 44, 20, 25, 49, 43, 25, 20, 25, 29, 27, 47, 48, 43, 45, 23, 26, 20, 43, 47, 29, 26, 44, 47, 21, 23, 46, 22, 28, 26, 49, 45, 49, 48, 45, 48, 43, 27, 24, 24, 23, 25, 22, 28, 41, 40, 20, 29, 41, 21, 22, 47, 28, 44, 21, 49, 47, 48, 24, 49, 21, 23, 26, 22, 46, 49, 29, 22, 40, 28, 42, 20, 23, 48, 28, 40, 28, 25, 25, 49, 24, 21, 27, 27, 40, 40, 24, 41, 21, 25, 28, 45, 40, 40, 41, 45, 45, 41, 40, 49, 28, 44, 41, 45, 48, 41, 49, 41, 42, 46, 45, 47, 28, 20, 49, 42, 25],
    'Income': [36, 81, 37, 35, 34, 85, 77, 82, 80, 77, 79, 81, 30, 29, 36, 73, 30, 81, 32, 82, 78, 40, 35, 29, 38, 32, 73, 36, 84, 80, 76, 80, 78, 30, 35, 82, 29, 85, 33, 76, 34, 76, 32, 30, 31, 74, 80, 31, 84, 85, 37, 83, 80, 31, 80, 33, 77, 30, 76, 33, 38, 77, 85, 28, 30, 34, 30, 33, 31, 74, 75, 76, 33, 33, 83, 40, 74, 76, 73, 76, 33, 36, 40, 74, 77, 37, 31, 33, 35, 77, 37, 28, 73, 81, 34, 31, 81, 78, 77, 82, 35, 74, 36, 39, 33, 78, 77, 84, 33, 74, 31, 30, 81, 82, 31, 78, 76, 31, 85, 34, 31, 80, 73, 35, 33, 78, 35, 32, 75, 80, 77, 79, 31, 31, 78, 77, 29, 82, 40, 82, 77, 29, 83, 81, 29, 75, 75, 81, 80, 33, 33, 40, 77, 78, 29, 74, 77, 34, 80, 31, 79, 82, 32, 34, 75, 39, 81, 82, 36, 81, 78, 74, 39, 83, 75, 76, 76, 77, 74, 79, 77, 33, 34, 36, 81, 34, 82, 35, 78, 80, 79, 78, 83, 34, 77, 80, 39, 75, 32, 77, 35, 77, 39, 31, 34, 81, 33, 34, 31, 83, 33, 39, 30, 39, 82, 32, 82, 82, 78, 31, 77, 29, 75, 77, 80, 75, 77, 77, 79, 75, 31, 37, 78, 85, 77, 36, 81, 31, 32, 33, 36, 32, 76, 80, 83, 75, 81, 39, 74, 80, 30, 76, 38, 80, 33, 83, 78, 31, 38, 32, 38, 34, 40, 37, 29, 85, 82, 30, 75, 80, 74, 83, 39, 70, 35, 78, 79, 79, 33, 71, 35, 75, 35, 74, 31, 31, 84, 31, 78, 86, 76, 79, 71, 82, 78, 73, 72, 86, 81, 40, 37, 79, 71, 78, 76, 74, 77, 79, 78, 32, 79, 32, 79, 73, 37, 77, 37, 83, 85, 31, 75, 37, 78, 31, 32, 83, 74, 31, 79, 75, 77, 40, 35, 32, 33, 30, 32, 39, 37, 32, 79, 38, 78, 81, 77, 38, 33, 38, 28, 77, 34, 29, 36, 29, 32, 76, 38, 83, 39, 76, 37, 32, 81, 80, 77, 37, 70, 35, 31, 80, 75, 31, 31, 73, 81, 71, 40, 35, 37, 28, 34, 35, 77, 37, 28, 33, 86, 32, 38, 76, 36, 31, 34, 36, 30, 77, 79, 77, 32, 38, 38, 79, 35, 31, 77, 29, 80, 79, 80, 38, 31, 35, 36, 34, 35, 34, 79, 34, 81, 34, 39, 36, 77, 82, 70, 78, 34, 83, 33, 31, 76, 77, 79, 74, 80, 76, 79, 76, 37, 79, 81, 40, 73, 32, 35, 78, 35, 34, 30, 33, 76, 33, 77, 29, 39, 35, 34, 72, 78, 29, 36, 78, 36, 38, 35, 78, 77, 86, 37, 31, 82, 77, 80, 81, 31, 76, 35, 74, 37, 81, 76, 74, 77, 33, 79, 36, 31, 31, 32, 74, 36, 35, 35, 79, 77, 73, 77, 33, 34, 80, 39, 34, 38, 77, 85, 31, 82, 81, 76, 83, 79, 33, 79, 37, 31, 80, 29, 28, 37, 78, 78, 38, 79, 83, 76, 79, 35, 33, 78, 82, 31, 35, 85, 33, 80, 81, 37, 32, 77, 76, 29, 81, 34, 74, 33, 35, 31, 83, 82, 38, 71, 36, 34, 80, 73, 33, 72, 74, 33, 31, 72, 30, 30, 76, 34, 37, 78, 79, 37, 35, 84, 84, 81, 73, 40, 76, 74, 30, 75, 31, 82, 81, 81, 34, 37, 83, 38, 38, 31, 38, 38, 35, 77, 80, 37, 70, 75, 79, 83, 32, 71, 80, 36, 34, 79, 77, 75, 75, 78, 82, 35, 82, 76, 39, 37, 33, 37, 80, 76, 76, 80, 77, 80, 37, 77, 32, 36, 80, 30, 79, 30, 76, 35, 78, 83, 81, 82, 77, 38, 30, 31, 75, 37, 72, 84, 28, 34, 76, 34, 84, 77, 80, 75, 78, 38, 28, 73, 81, 35, 30, 76, 30, 79, 38, 78, 32, 77, 30, 30, 81, 33, 75, 29, 82, 31, 72, 30, 78, 35, 80, 72, 77, 37, 79, 29, 76, 37, 78, 31, 36, 85, 30, 36, 74, 30, 38, 36, 75, 35, 74, 32, 32, 31, 74, 35, 32, 72, 37, 75, 82, 34, 74, 33, 81, 72, 29, 77, 76, 76, 79, 73, 35, 82, 30, 76, 29, 31, 76, 71, 77, 37, 76, 32, 34, 36, 74, 76, 33, 34, 80, 30, 72, 32, 78, 38, 71, 37, 84, 76, 38, 37, 75, 29, 83, 29, 78, 70, 79, 75, 77, 82, 33, 33, 32, 76, 77, 76, 79, 33, 34, 77, 34, 38, 76, 36, 77, 32, 78, 77, 82, 72, 37, 77, 75, 74, 35, 77, 37, 77, 36, 29, 77, 33, 79, 75, 77, 37, 34, 76, 80, 34, 76, 72, 77, 31, 32, 39, 83, 76, 77, 28, 77, 83, 73, 85, 31, 35, 84, 35, 33, 71, 81, 28, 78, 32, 82, 33, 78, 32, 31, 31, 86, 40, 32, 79, 76, 38, 37, 80, 71, 76, 78, 81, 38, 76, 31, 33, 32, 84, 33, 35, 79, 33, 35, 84, 30, 40, 36, 76, 74, 74, 32, 38, 77, 28, 35, 82, 76, 36, 37, 38, 78, 80, 77, 79, 39, 70, 80, 82, 83, 82, 35, 34, 79, 37, 79, 78, 37, 30, 81, 81, 83, 29, 84, 80, 76, 32, 74, 80, 30, 77, 82, 32, 77, 34, 36, 84, 79, 78, 84, 36, 75, 32, 80, 75, 76, 30, 70, 82, 81, 74, 38, 81, 34, 30, 33, 34, 33, 30, 82, 39, 36, 39, 82, 77, 34, 81, 34, 30, 81, 33, 72, 78, 34, 32, 75, 39, 28, 32, 77, 75, 36, 79, 83, 81, 80, 77, 35, 36, 37, 75, 74, 39, 76, 33, 32, 35, 79, 71, 34, 33, 37, 32, 77, 85, 39, 30, 78, 75, 37, 36, 32, 81, 74, 83, 29, 38, 79, 75, 37, 80, 75, 33, 77, 32, 29, 35, 33, 31, 35, 35, 75, 36, 34, 80, 79, 80, 35, 77, 37, 32, 38, 32, 76, 85, 36, 38, 33, 77, 37, 36, 79, 77, 38, 83, 78, 35, 77, 28, 72, 38, 30, 38, 77, 38, 79, 29, 37, 75, 34, 38, 79, 38, 31, 39, 37, 72, 78, 32, 77, 35, 82, 37, 37, 33, 36, 39, 39, 78, 37, 80, 33, 81, 79, 82, 28, 37, 71, 36, 80, 37, 72, 78, 32, 85, 39, 86, 78, 35, 35, 31, 76, 34, 81, 75, 28, 33, 86, 76, 36, 30, 35, 38, 37, 79, 82, 75, 78, 32, 34, 30, 77, 77, 38, 37, 78, 82, 30, 33, 76, 31, 37, 35, 79, 77, 84, 82, 77, 79, 78, 38, 34, 35, 32, 33, 31, 38, 77, 76, 29, 40, 77, 29, 33, 82, 37, 81, 31, 83, 84, 79, 32, 80, 31, 31, 34, 33, 81, 83, 40, 33, 71, 38, 73, 31, 33, 83, 38, 74, 38, 33, 34, 79, 33, 32, 38, 36, 76, 72, 34, 72, 32, 35, 37, 82, 71, 73, 78, 76, 80, 76, 71, 81, 37, 80, 73, 79, 84, 76, 82, 73, 72, 81, 78, 80, 37, 29, 81, 78, 36],
    'Churn': [0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0]
})


In [3]:
X=df[["Age","Income"]]
y=df["Churn"]
X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
    
)

In [6]:
models = {
    "Random Forest": RandomForestClassifier(
        n_estimators=100, random_state=42
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=100, random_state=42
    )
}

In [7]:
for name , model in models.items():
    model.fit(X_train,y_train)
    predictions=model.predict(X_test)
    cv_scores=cross_val_score(
        model,
        X,
        y,
        cv=5,
        scoring="accuracy"
    )
    print(f"\n{name}")
    print("Test accuracy:", accuracy_score(y_test, predictions))
    print("CV scores:", cv_scores)
    print("Mean CV accuracy:", cv_scores.mean())


Random Forest
Test accuracy: 1.0
CV scores: [1. 1. 1. 1. 1.]
Mean CV accuracy: 1.0

Extra Trees
Test accuracy: 1.0
CV scores: [1. 1. 1. 1. 1.]
Mean CV accuracy: 1.0
